# 002 · Strings

> **Syntax › Data Types** · Python 3.14

Python's `str` is an **immutable sequence of Unicode code points** — very close to C#'s `string`. Most of what you know carries over; the differences are in the literal syntax, the method names, and the fact that a string is also a full sequence you can index, slice, and loop over.

**In this notebook you will learn:**

- the many ways to write string literals: quotes, triple quotes, raw strings, escapes
- indexing (including negative indexes) and basic slicing
- the most common string methods and their `System.String` counterparts
- why there is no `char` type, and how `ord`/`chr` replace `(int)c`/`(char)n`
- how to build strings efficiently (`join` instead of `StringBuilder`)

## C# mental model

| C# | Python |
|---|---|
| `string` | `str` |
| `char` | no separate type — a `str` of length 1 |
| `"..."` only | `'...'` or `"..."` (identical) |
| `@"C:\path"` verbatim string | `r"C:\path"` raw string |
| `"""..."""` raw string literal (C# 11) | `"""..."""` triple-quoted string |
| `s.Length` | `len(s)` |
| `StringBuilder` | `"".join(parts)` |

Like C#, strings are **immutable**: every "modifying" method returns a new string.

## String literals

Single and double quotes are completely interchangeable — pick one (PEP 8 doesn't mandate either; many teams follow the formatter *Black*, which prefers double quotes). Having both makes it easy to embed the other kind without escaping.

```csharp
// C#
var a = "hello";
var b = "She said \"hi\"";
```

In [ ]:
a = 'hello'
b = "hello"
print(a == b)                     # True — no difference at all

print("It's easy")                # apostrophe inside double quotes
print('She said "hi"')            # double quotes inside single quotes
print("She said \"hi\"")          # escaping works too

Triple quotes (`'''` or `"""`) create multi-line strings. They keep newlines and indentation exactly as written — similar to C# 11 raw string literals, but **escape sequences are still processed** and indentation is *not* stripped automatically. Use `textwrap.dedent` to remove common leading whitespace.

```csharp
// C# 11
var sql = """
    SELECT *
    FROM orders
    """;   // C# strips the indentation up to the closing quotes
```

In [ ]:
import textwrap

sql = """
    SELECT *
    FROM orders
"""
print(repr(sql))                    # indentation and newlines are kept
print(textwrap.dedent(sql).strip()) # dedent + strip ≈ what C# 11 does for you

## Escape sequences and raw strings

The usual escapes work: `\n`, `\t`, `\\`, `\"`, `\'`, plus `\uXXXX`, `\U0001F600`, and `\N{NAME}` (Unicode name).

A **raw string** `r"..."` turns off escape processing — Python's version of C#'s verbatim `@"..."`. It is used mainly for Windows paths and regular expressions.

```csharp
// C#
var path  = @"C:\new\table";
var regex = @"\d+\.\d+";
```

In [ ]:
print("tab:\tend")
print("caf\u00e9", "\N{GREEK SMALL LETTER ALPHA}", "\U0001F40D")

print("C:\new\table")     # oops: \n and \t were interpreted!
print(r"C:\new\table")    # raw string: backslashes kept
print(r"\d+\.\d+")        # typical regex pattern

⚠️ **Gotcha:** a raw string cannot end with an odd number of backslashes — `r"C:\folder\"` is a syntax error. Also, unlike `@"..."`, you can't double a quote to escape it; use the other quote type instead.

## Length, indexing, and slicing

`len(s)` replaces `s.Length`. Indexing works like C#, and negative indexes count from the end (C#: `s[^1]`).

**Slicing** `s[start:stop]` extracts a substring — like C#'s `s[start..stop]` range or `Substring`. The stop index is exclusive. *(Slicing gets a full treatment in Collections › 005 · Slicing.)*

```csharp
// C#
var s = "Python";
var last  = s[^1];        // 'n'
var first3 = s[..3];      // "Pyt"
var sub   = s.Substring(2, 3);  // "tho"
```

In [ ]:
s = "Python"

print(len(s))       # 6
print(s[0], s[-1])  # P n
print(s[:3])        # Pyt   — first 3 characters
print(s[2:5])       # tho   — index 2 up to (not including) 5
print(s[-3:])       # hon   — last 3 characters
print(s[::-1])      # nohtyP — reversed (step of -1)

Indexing out of range raises `IndexError` — but slicing never does, it just clamps:

In [ ]:
try:
    s[10]
except IndexError as e:
    print(f"{type(e).__name__}: {e}")

print(repr(s[3:100]))   # 'hon' — slices are forgiving

Strings are immutable, so you cannot assign to an index (same as C#):

In [ ]:
try:
    s[0] = "J"
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("J" + s[1:])   # build a new string instead

## No `char` type: `ord` and `chr`

Indexing a string gives you another `str` of length 1. To get the code point use `ord()` (C#: `(int)c`), and to go back use `chr()` (C#: `(char)n`).

In [ ]:
c = "A"[0]
print(type(c), len(c))     # <class 'str'> 1
print(ord("A"))            # 65
print(chr(97))             # a
print(ord("€"), chr(0x1F40D))

⚠️ **Gotcha:** a C# `char` is a UTF-16 code unit, so emoji take two `char`s and `"🐍".Length == 2`. Python's `str` is indexed by **code point**, so `len("🐍") == 1`.

In [ ]:
print(len("🐍"))                   # 1 (C#: 2)
print(len("🐍".encode("utf-8")))   # 4 bytes in UTF-8
print(len("café"))                 # 4

## Common methods

| C# | Python |
|---|---|
| `s.ToUpper()` / `ToLower()` | `s.upper()` / `s.lower()` |
| `s.Trim()` / `TrimStart()` / `TrimEnd()` | `s.strip()` / `lstrip()` / `rstrip()` |
| `s.Split(',')` | `s.split(",")` |
| `string.Join(", ", items)` | `", ".join(items)` |
| `s.Replace("a", "b")` | `s.replace("a", "b")` |
| `s.StartsWith(x)` / `EndsWith(x)` | `s.startswith(x)` / `s.endswith(x)` |
| `s.Contains(x)` | `x in s` |
| `s.IndexOf(x)` | `s.find(x)` (returns -1) or `s.index(x)` (raises) |
| `string.IsNullOrWhiteSpace(s)` | `not s.strip()` (or `s.isspace()` for non-empty) |
| `s.PadLeft(5)` / `PadRight(5)` | `s.rjust(5)` / `s.ljust(5)` |

In [ ]:
text = "  Hello, World  "

print(repr(text.strip()))           # 'Hello, World'
print(text.upper(), text.lower())
print(text.strip().replace("World", "Python"))
print(text.strip().startswith("Hello"), text.strip().endswith("!"))
print("hello world".title(), "Hello".swapcase(), "hello".capitalize())

`strip()` can also take a set of characters to remove, like `Trim(params char[])`:

In [ ]:
print("xxhixx".strip("x"))        # hi
print("$$$42.00".lstrip("$"))     # 42.00
print("report.txt".removesuffix(".txt"))   # report — removes exact suffix (3.9+)

## Splitting and joining

`split()` with **no argument** splits on any run of whitespace and drops empty parts — a very handy default that C# does not have built in (`Split((char[])null, StringSplitOptions.RemoveEmptyEntries)` comes close).

⚠️ **Gotcha:** `join` is a method on the **separator**, not on the list: `", ".join(items)`, not `items.join(", ")`.

In [ ]:
print("a,b,,c".split(","))             # ['a', 'b', '', 'c']
print("  one  two\tthree\n".split())   # ['one', 'two', 'three']
print("k=v=w".split("=", maxsplit=1))  # ['k', 'v=w']
print("line1\nline2\r\nline3".splitlines())

words = ["red", "green", "blue"]
print(", ".join(words))                # C#: string.Join(", ", words)

`join` only accepts strings — convert other items first:

In [ ]:
nums = [1, 2, 3]
try:
    print("-".join(nums))
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("-".join(str(n) for n in nums))   # 1-2-3

## Searching

`in` replaces `Contains`. For positions, `find` returns `-1` when not found (like `IndexOf`), while `index` raises `ValueError`.

In [ ]:
s = "banana"

print("nan" in s)          # True   (C#: s.Contains("nan"))
print("xyz" not in s)      # True
print(s.find("an"))        # 1      (C#: s.IndexOf("an"))
print(s.rfind("an"))       # 3      (C#: s.LastIndexOf("an"))
print(s.find("xyz"))       # -1
print(s.count("a"))        # 3

try:
    s.index("xyz")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Character-class checks replace `char.IsDigit`, `char.IsLetter` etc. — and they work on whole strings:

In [ ]:
print("12345".isdigit(), "abc".isalpha(), "abc123".isalnum())
print("   ".isspace(), "Hello".istitle(), "ABC".isupper())
print("".isdigit())    # False — empty strings are False for all is*() checks

## Concatenation, repetition, and building strings

`+` concatenates and `*` repeats. Because strings are immutable, building a long string with `+=` in a loop creates many intermediate strings — the C# answer is `StringBuilder`; the Python answer is to collect parts in a **list** and `join` them once.

```csharp
// C#
var sb = new StringBuilder();
foreach (var i in Enumerable.Range(0, 5)) sb.Append(i).Append(',');
var result = sb.ToString();
```

In [ ]:
print("ab" + "cd")         # abcd
print("-" * 20)            # repetition (C#: new string('-', 20))

parts = []
for i in range(5):
    parts.append(str(i))   # like sb.Append(...)
result = ",".join(parts)   # like sb.ToString()
print(result)

⚠️ **Gotcha:** `+` does not convert numbers for you — `"Total: " + 5` is a `TypeError` (C# would call `ToString()` implicitly). Use an f-string *(see Data Types › 003 · String Formatting)*.

In [ ]:
try:
    "Total: " + 5
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("Total: " + str(5))
print(f"Total: {5}")

## Comparing strings

`==` compares by value (like C#'s `==` on strings). `<`, `>` compare **lexicographically by code point** — this is ordinal comparison, *not* culture-aware, so uppercase sorts before lowercase.

For case-insensitive equality use `casefold()` (more thorough than `lower()` for some languages) — the closest analogue of `StringComparison.OrdinalIgnoreCase`.

In [ ]:
print("apple" == "apple")       # True
print("apple" < "banana")       # True
print("Zebra" < "apple")        # True — 'Z' (90) < 'a' (97)
print("Straße".casefold() == "STRASSE".casefold())   # True
print("Straße".lower() == "STRASSE".lower())         # False

## Iterating

A string is iterable — a `for` loop yields each character (as a 1-length `str`):

In [ ]:
for ch in "abc":
    print(ch, ord(ch))

vowels = sum(1 for ch in "programming" if ch in "aeiou")
print("vowels:", vowels)

## Summary

| Task | C# | Python |
|---|---|---|
| Length | `s.Length` | `len(s)` |
| Last char | `s[^1]` | `s[-1]` |
| Substring | `s[2..5]` / `s.Substring(2, 3)` | `s[2:5]` |
| Verbatim / raw | `@"C:\x"` | `r"C:\x"` |
| Multi-line | `"""..."""` | `"""..."""` (+ `textwrap.dedent`) |
| Contains | `s.Contains(x)` | `x in s` |
| Find | `s.IndexOf(x)` | `s.find(x)` / `s.index(x)` |
| Trim | `s.Trim()` | `s.strip()` |
| Split | `s.Split(',')` | `s.split(",")` |
| Join | `string.Join(",", xs)` | `",".join(xs)` |
| Char code | `(int)c` / `(char)n` | `ord(c)` / `chr(n)` |
| Repeat | `new string('-', 3)` | `"-" * 3` |
| Build | `StringBuilder` | list + `"".join(parts)` |
| Ignore case | `OrdinalIgnoreCase` | `a.casefold() == b.casefold()` |

## Exercises

1. Write `is_palindrome(s)` that ignores case and any non-letter characters. `"A man, a plan, a canal: Panama"` should return `True`.
2. Given `csv = "  alice , bob,carol ,  dave "`, produce the list `['alice', 'bob', 'carol', 'dave']`.
3. Write a Caesar cipher `shift("abc xyz", 3)` → `"def abc"` using `ord`/`chr` (lowercase letters only; leave other characters unchanged).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def is_palindrome(s: str) -> bool:
    letters = "".join(ch.casefold() for ch in s if ch.isalpha())
    return letters == letters[::-1]

print(is_palindrome("A man, a plan, a canal: Panama"))   # True

# 2
csv = "  alice , bob,carol ,  dave "
names = [part.strip() for part in csv.split(",")]
print(names)

# 3
def shift(text: str, n: int) -> str:
    out = []
    for ch in text:
        if "a" <= ch <= "z":
            out.append(chr((ord(ch) - ord("a") + n) % 26 + ord("a")))
        else:
            out.append(ch)
    return "".join(out)

print(shift("abc xyz", 3))   # def abc
```

</details>